# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — "What Predicts Health?" (ML appendix, feature importance)

**The claim.** A Random Forest predicting health score ranks Average Position first
at 43% importance, then Impressions at 32% and Scroll Depth at 15%.

**Where does the label come from?** Health score is a FlyRank composite, defined in
the methodology as impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll
depth (20 pts). So the target is constructed from the same measurements the model
receives as features. Position and impressions together make up 60 of the score's
100 points, and they account for 75% of the model's reported importance.

**Does the validation design carry the claim?** The paper is upfront about this — it
states that the target is partly constructed from these inputs, that importance is
descriptive rather than causal, and that high importance is expected. That disclosure
is what makes the section usable rather than misleading, and it is the standard I
want to hold my own appendix to.

**What I would want to see before acting on it.** The ranking currently tells me how
the score is built, which I already know from its definition. The question it cannot
answer is which measurable page properties predict health *independently* of the
score's own arithmetic. Refitting with position and impressions removed from the
feature set would separate those two things. I would also want to know whether the
80/20 split groups by brand — with 57 brands in the portfolio, a random row split
would let the model recognise brands rather than learn page-level patterns, which is
exactly the mistake I made in my own Week-2 notebook.

## Finding 2 — Finding #4, "The Freshness Multiplier"

**The claim.** The 31-90 day freshness window is the strongest measured growth band
at 7.88:1 growth-to-decline. Separately, 365+ day content refreshed within 30 days
shows a 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4,039).

**Where does the label come from?** Growth and decline come from a 30-day-versus-
previous-30-day impression comparison, and the trend definition in this paper uses a
+/-10% threshold. The freshness buckets come from days since last update.

**Does the validation design carry the claim?** For the 31-90 band, the paper reports
the ratio and the sample is large enough to read. For the 361+ bucket it demotes its
own number explicitly: 283:1 rests on 283 growing pages against a single declining
one, and the paper says so in the chart note rather than leading with it. That is the
same discipline I tried to apply when I demoted my own position column after finding
values below 1.

**What I would want to see before acting on it.** The 3.2x and 57x figures are used
as headline numbers in the same section where the 283:1 is set aside, and I cannot
tell from the page how many pages sit behind them. The refreshed-versus-unrefreshed
comparison is also observational: pages that get refreshed are chosen by someone, and
pages worth refreshing are plausibly the ones already performing better. A sample size
per cell, and a note on how refresh candidates are selected, would tell me whether the
lift is the refresh or the selection. The paper's own limitations section makes the
general version of this point — correlations do not prove causation — so this is a
question about where that caveat applies, not a disagreement with it.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected")

Connected


In [2]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,
        f.has_ga4,
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,
        COALESCE(m.imp_mar, 0) AS imp_mar
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,trend_within_feb,has_ga4,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,0.662,0,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,1.076,0,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,0.833,0,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,NaN,0,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,0.451,0,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [3]:
df = features.copy()

FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS
df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"Rows: {len(df):,}")
print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")

Rows: 93,654
Positive rate: 27.6% (25,887 declining)


In [4]:
encoded_prefixes = ("ctype_", "intent_")
df = df.drop(columns=[c for c in df.columns if c.startswith(encoded_prefixes)], errors="ignore")
df = df.drop(columns=["competition_level_ord"], errors="ignore")

competition_order = {"unknown": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["competition_level_ord"] = df["competition_level"].map(competition_order).fillna(0).astype(int)

one_hot = pd.get_dummies(df[["content_type", "main_intent"]],
                         prefix=["ctype", "intent"], dtype=int)
df = pd.concat([df, one_hot], axis=1)

drop_cols = ["client_hash_id", "content_hash_id", "imp_mar", "is_declining",
             "content_type", "competition_level", "main_intent"]
model_features = [c for c in df.columns if c not in drop_cols]

print(f"Model-ready features: {len(model_features)}")

Model-ready features: 23


In [5]:
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score


def precision_at_k(y_true, scores, k=50):
    top = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top].mean())


X = df[model_features]
y = df["is_declining"]
groups = df["client_hash_id"]

splits = {
    "Random split (naive)": StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y),
    "Grouped by client (honest)": GroupKFold(n_splits=5).split(X, y, groups),
}

audit = {}
for name, split in splits.items():
    scores = {"auc": [], "ap": [], "p50": []}
    for tr, te in split:
        model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                       random_state=42, n_jobs=-1)
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        scores["auc"].append(roc_auc_score(y.iloc[te], p))
        scores["ap"].append(average_precision_score(y.iloc[te], p))
        scores["p50"].append(precision_at_k(y.iloc[te], p))
    audit[name] = scores

before_after = pd.DataFrame({
    name: {
        "ROC AUC": f"{np.mean(s['auc']):.3f} +/- {np.std(s['auc']):.3f}",
        "Avg precision": f"{np.mean(s['ap']):.3f} +/- {np.std(s['ap']):.3f}",
        "Precision@50": f"{np.mean(s['p50']):.3f} +/- {np.std(s['p50']):.3f}",
    }
    for name, s in audit.items()
}).T

print(f"Base rate: {y.mean():.3f}\n")
print(before_after.to_string())

print("\nDifference (random minus grouped):")
for metric, key in [("ROC AUC", "auc"), ("Avg precision", "ap"), ("Precision@50", "p50")]:
    diff = np.mean(audit["Random split (naive)"][key]) - np.mean(audit["Grouped by client (honest)"][key])
    print(f"  {metric:15} {diff:+.3f}")

Base rate: 0.276

                                    ROC AUC    Avg precision     Precision@50
Random split (naive)        0.794 +/- 0.003  0.668 +/- 0.005  1.000 +/- 0.000
Grouped by client (honest)  0.685 +/- 0.045  0.486 +/- 0.161  0.836 +/- 0.142

Difference (random minus grouped):
  ROC AUC         +0.109
  Avg precision   +0.182
  Precision@50    +0.164


**Before and after: the same model, two split designs.**

| | ROC AUC | Avg precision | Precision@50 |
|---|---|---|---|
| Random split (naive) | 0.794 +/- 0.003 | 0.668 +/- 0.005 | 1.000 +/- 0.000 |
| Grouped by client (honest) | 0.685 +/- 0.045 | 0.486 +/- 0.161 | 0.836 +/- 0.142 |
| Difference | +0.109 | +0.182 | +0.164 |

Same Random Forest, same 93,654 pages, same label, same 5 folds. Only the split
design changes.

**Precision@50 of 1.000 is the finding, not the achievement.** Under the random split
the model puts 50 declining pages at the top of the queue in every fold, with zero
variance. A perfect score with no spread across five folds is a warning sign rather
than a result: it means the five folds are not really five experiments.

**Why it happens.** A random row split scatters each client's pages across train and
test. My 93,654 pages belong to 55 clients, and pages from one client share a site, a
topic mix, and a traffic pattern. The model sees some of a client's February pages
with their March outcomes, then predicts other pages from the same client in the same
period. It can recognise which clients had a bad March instead of learning why pages
decline.

**The variance tells the same story.** Under the random split the fold-to-fold spread
is near zero (+/- 0.003 ROC AUC); under the grouped split it is +/- 0.045, and
Precision@50 swings +/- 0.142. The honest design exposes how much the answer depends
on which clients you happen to hold out — which is real uncertainty that the random
split was hiding, not noise the grouped split introduced.

**What I take from it.** The grouped numbers are the ones I report. The observed gap
of +0.109 ROC AUC is the size of the overstatement I would have published had I used
the default split — a directional measure of how much a validation choice, not a
modelling choice, can move a headline number.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 1. Correlation with the label
correlations = df[model_features + ["is_declining"]].corr()["is_declining"].drop("is_declining")
correlations = correlations.reindex(correlations.abs().sort_values(ascending=False).index)
print("Correlation with the label, strongest first")
print(correlations.head(8).round(4).to_string())
print(f"Max absolute correlation: {correlations.abs().max():.3f}\n")

# 2. Single-feature AUC
X_tr, X_te, y_tr, y_te = train_test_split(
    df[model_features], df["is_declining"], test_size=0.3, random_state=42,
    stratify=df["is_declining"]
)
single = {}
for col in model_features:
    tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr[[col]], y_tr)
    single[col] = roc_auc_score(y_te, tree.predict_proba(X_te[[col]])[:, 1])
single = pd.Series(single).sort_values(ascending=False)
print("Single-feature ROC AUC, strongest first")
print(single.head(8).round(4).to_string())
print(f"Features above 0.90 alone: {(single > 0.90).sum()}\n")

# 3. Window audit: every feature must come from February or dim_content
FEATURE_WINDOW, LABEL_WINDOW = "2026-02", "2026-03"
audit = {c: FEATURE_WINDOW for c in
         ["imp_feb", "clicks_feb", "days_visible_feb", "ctr_feb",
          "imp_per_active_day", "trend_within_feb", "no_h1_impressions", "has_ga4"]}
for c in model_features:
    if c not in audit:
        audit[c] = "dim_content snapshot"

undocumented = [c for c in model_features if c not in audit]
march_sourced = [c for c, src in audit.items() if src == LABEL_WINDOW]

print(f"Features audited    : {len(model_features)}")
print(f"Undocumented source : {len(undocumented)}")
print(f"Sourced from March  : {len(march_sourced)}")
assert not undocumented, "Every feature must have a documented source window"
assert not march_sourced, "No feature may come from the label window"

# 4. Control: plant a known leak and confirm the detector reacts
leak_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
leak_tree.fit(df.loc[X_tr.index, ["imp_mar"]], y_tr)
leak_auc = roc_auc_score(y_te, leak_tree.predict_proba(df.loc[X_te.index, ["imp_mar"]])[:, 1])
print(f"\nControl — imp_mar alone: ROC AUC {leak_auc:.3f}")
print("Audit passed: no feature draws on the March label window.")

Correlation with the label, strongest first
no_h1_impressions      -0.1242
intent_unknown          0.1015
ctype_feedly article    0.0837
competition             0.0733
has_ga4                -0.0711
ctr_feb                -0.0673
word_count             -0.0644
intent_informational   -0.0532
Max absolute correlation: 0.124

Single-feature ROC AUC, strongest first
trend_within_feb      0.6793
days_visible_feb      0.6366
content_age_days      0.6296
word_count            0.6088
clicks_feb            0.5834
ctr_feb               0.5793
imp_per_active_day    0.5783
imp_feb               0.5604
Features above 0.90 alone: 0

Features audited    : 23
Undocumented source : 0
Sourced from March  : 0

Control — imp_mar alone: ROC AUC 0.767
Audit passed: no feature draws on the March label window.


**Leakage audit on the final 23-feature set.**

Three checks plus a control, run on the same feature set the Week-5 model uses.

**Check 1 — correlation with the label.** The strongest correlation is
`no_h1_impressions` at -0.124, and no feature exceeds 0.124 in absolute terms.
Nothing is close to the range where a feature would be reading the answer.

**Check 2 — single-feature ROC AUC.** No feature scores above 0.90 on its own. The
strongest is `trend_within_feb` at 0.679, followed by `days_visible_feb` (0.637) and
`content_age_days` (0.630). These are the values I would expect from real but
imperfect signals on a hard problem.

**Check 3 — the window audit.** All 23 features are documented as coming from the
February 2026 partition or from `dim_content` metadata. Zero are sourced from the
March label window, and zero are undocumented. Both conditions are enforced with
`assert` statements rather than printed claims, so the notebook fails loudly if a
future edit breaks them.

**The control confirms a known blind spot.** Scoring `imp_mar` alone — the label's own
ingredient — gives ROC AUC 0.767, which sits *below* my 0.90 screening threshold. A
column that leaks the answer would have passed the single-feature check undetected.

The reason is that the label is a ratio: March impressions against February
impressions. March alone is ambiguous, because 500 impressions is a collapse for a
large page and growth for a small one. The leak only becomes visible once the February
baseline sits alongside it.

Measured implication: single-feature screening is necessary but not sufficient for
ratio labels. The window audit is what would catch such a feature, because it tests
where a column comes from rather than how well it predicts. A detector that finds
nothing is only reassuring once you have shown it can find something — which is why
the control runs alongside the checks rather than instead of them.

**Two exclusions worth restating, both decided before modelling.**

`content_updated_date` and `last_optimized_date` are excluded because `dim_content` is
a current-state snapshot rather than history. A page updated in March carries a March
date, so those fields can encode the label window while looking like ordinary
metadata. They are the most dangerous columns in the release precisely because they
would raise no suspicion in a feature list.

The position columns are excluded on separate evidence. Inspecting one page day by day
showed `gsc_avg_position` values below 1 — for example 3,519 impressions against a
`gsc_sum_position` of 524, giving 0.149. Position 1 is the best rank Google awards, so
a value below 1 is not a real position. That is a data-quality exclusion, not a
leakage one, but it belongs in the same audit because both are reasons a column cannot
be trusted as a feature.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**The sentence I am rewriting.**

From my Week-5 notebook:

> "Random Forest is ahead. Precision@50 rises from 0.680 to 0.836, and ROC AUC from
> 0.633 to 0.685."

**Why it goes further than the evidence.** The means do differ, but the fold-to-fold
spreads overlap substantially: the hand rule ranges roughly 0.45 to 0.91 on
Precision@50, the Random Forest roughly 0.69 to 0.98. In some client mixes the rule
may have matched or beaten the model. Reporting only the means presents a comparison
as settled when five folds cannot settle it.

**Rewritten.**

> Across 5 client-grouped folds, the Random Forest ranking **observed** a higher mean
> Precision@50 than the hand rule (0.836 against 0.680) and a higher mean ROC AUC
> (0.685 against 0.633). The spreads overlap, so this is a **directional** result
> rather than a demonstrated win. The model was also the more stable of the two
> (+/- 0.045 ROC AUC against the rule's +/- 0.085), which is a second reason to prefer
> it. For **decision-support** I would use the model's ranking while keeping the hand
> rule available, since the two are close enough that losing the model would not be a
> crisis.

**A second claim that needed the same treatment.** My Week-5 error analysis reported
Precision@50 of 0.836 alongside 300 true positives against 4,651 false negatives on
held-out clients — a recall of roughly 6%. Both numbers are **measured**, but quoting
the first without the second would overstate what the model does.

> The ranking is reliable at the top of the queue and blind to most of the declining
> population. The misses share an **observed** profile: median `trend_within_feb` of
> 1.182, meaning pages that were growing through February and then turned in March. A
> February-only feature window cannot see that coming, so this is a limitation of the
> window I chose rather than of the model class.

**And one this notebook forced me to add.** Before running section 2 I would have
described my Week-5 result without reference to the split design. The **measured** gap
between a random split and a client-grouped split on the same model is +0.109 ROC AUC,
+0.182 average precision, and +0.164 Precision@50 — and the random split produced a
perfect Precision@50 of 1.000 with zero variance across folds.

> Any headline number I publish is a claim about a validation design as much as about
> a model. Stating the split alongside the metric is not a caveat; without it the
> number is not interpretable.

**The rule I applied throughout.** Every claim is tied to something measured on a
stated split. Where I wanted to write "beats" or "proves", I wrote observed or
directional instead. Where a number could mislead alone, I paired it with the number
that qualifies it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.